# Final Project Part 3 Analysis Notebook

Group Member: Changrong Wu

This notebook creates the central interactive visualization and two contextual visualizations used in the Jekyll data journalism article.

In [20]:
import pandas as pd
import altair as alt
import json

alt.data_transformers.disable_max_rows()

# =========================
# 1. Load main dataset and map boundary file
# =========================
# Put these two files in the same folder as this notebook:
# chicago_crime_2years.csv
# Boundaries_-_Community_Areas_20260501.geojson

df = pd.read_csv("chicago_crime_1years.csv")

with open("Boundaries_-_Community_Areas_20260501.geojson", "r") as f:
    chicago_geojson = json.load(f)
community_areas = chicago_geojson["features"]

# =========================
# 2. Clean and prepare data
# =========================

df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y %I:%M:%S %p", errors="coerce")
if df["Date"].isna().mean() > 0.5:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

df = df.dropna(subset=["Date", "Primary Type", "Latitude", "Longitude", "Community Area"])
df["Community Area"] = pd.to_numeric(df["Community Area"], errors="coerce")
df = df.dropna(subset=["Community Area"])
df["Community Area"] = df["Community Area"].astype(int)

# =========================
# 3. Keep only one year of data
# =========================

latest_date = df["Date"].max()
start_date = latest_date - pd.DateOffset(years=1)

df_1yr = df[df["Date"] >= start_date].copy()

df_1yr["YearMonth"] = df_1yr["Date"].dt.to_period("M").dt.to_timestamp()
df_1yr["Hour"] = df_1yr["Date"].dt.hour
df_1yr["DayOfWeek"] = df_1yr["Date"].dt.day_name()

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# =========================
# 4. Build community lookup from GeoJSON
# =========================

community_lookup = []

for feature in community_areas:
    props = feature["properties"]
    area_number = (
        props.get("area_numbe")
        or props.get("area_num_1")
        or props.get("area_num")
        or props.get("AREA_NUMBE")
        or props.get("AREA_NUM_1")
    )
    area_name = (
        props.get("community")
        or props.get("community_area")
        or props.get("name")
        or props.get("COMMUNITY")
        or props.get("NAME")
    )

    if area_number is not None and area_name is not None:
        community_lookup.append({
            "Community Area": int(area_number),
            "Community Name": str(area_name).title()
        })

community_lookup_df = pd.DataFrame(community_lookup)

df_1yr = df_1yr.merge(community_lookup_df, on="Community Area", how="left")
df_1yr["Community Name"] = df_1yr["Community Name"].fillna("Unknown")

for feature in community_areas:
    props = feature["properties"]
    geo_name = (
        props.get("community")
        or props.get("community_area")
        or props.get("name")
        or props.get("COMMUNITY")
        or props.get("NAME")
        or "Unknown"
    )
    props["Community Name"] = str(geo_name).title()

# =========================
# 5. Keep top crime types
# =========================

top_crimes = df_1yr["Primary Type"].value_counts().head(10).index.tolist()
df_top = df_1yr[df_1yr["Primary Type"].isin(top_crimes)].copy()

# =========================
# 6. Option 4: Keep only necessary columns for the map
# =========================

map_cols = [
    "Date",
    "YearMonth",
    "Primary Type",
    "Latitude",
    "Longitude",
    "Community Name"
]

df_map_full = df_top[map_cols].dropna(subset=["Latitude", "Longitude"]).copy()

# Convert dates to lighter formats for HTML
df_map_full["Date"] = df_map_full["Date"].dt.strftime("%Y-%m-%d")
df_map_full["YearMonth"] = pd.to_datetime(df_map_full["YearMonth"])

# =========================
# 7. Option 1: Reduce number of map points
# =========================

MAP_SAMPLE_SIZE = 1500

if len(df_map_full) > MAP_SAMPLE_SIZE:
    df_map = df_map_full.sample(MAP_SAMPLE_SIZE, random_state=42).copy()
else:
    df_map = df_map_full.copy()

# =========================
# 8. Print summary
# =========================

print("Rows in final one-year dataset:", len(df_1yr))
print("Start date:", df_1yr["Date"].min())
print("End date:", df_1yr["Date"].max())
print("Top crime types:", top_crimes)
print("Rows used in map:", len(df_map))

Rows in final one-year dataset: 233456
Start date: 2025-04-09 00:00:00
End date: 2026-04-09 00:00:00
Top crime types: ['THEFT', 'BATTERY', 'CRIMINAL DAMAGE', 'ASSAULT', 'MOTOR VEHICLE THEFT', 'OTHER OFFENSE', 'DECEPTIVE PRACTICE', 'BURGLARY', 'NARCOTICS', 'ROBBERY']
Rows used in map: 1500


In [23]:
# =========================
# Central interactive visualization for public article
# GitHub-safe smaller HTML version
# =========================

# Aggregate monthly trend data first.
# This prevents Altair from embedding the full df_top dataset into the HTML.
trend_data = (
    df_top
    .groupby(["YearMonth", "Primary Type", "Community Name"])
    .size()
    .reset_index(name="Reports")
)

community_options = ["All"] + sorted(df_top["Community Name"].dropna().unique().tolist())

region_select = alt.param(
    name="SelectedRegion",
    bind=alt.binding_select(
        options=community_options,
        name="Choose a Chicago community area: "
    ),
    value="All"
)

region_filter = "(SelectedRegion == 'All') || (datum['Community Name'] == SelectedRegion)"
geo_region_filter = "(SelectedRegion == 'All') || (datum.properties['Community Name'] == SelectedRegion)"

time_brush = alt.selection_interval(encodings=["x"])
crime_type_select = alt.selection_point(
    fields=["Primary Type"],
    bind="legend",
    toggle=True
)

background_map = alt.Chart(alt.Data(values=community_areas)).mark_geoshape(
    stroke="gray",
    strokeWidth=0.8
).encode(
    fill=alt.condition(
        geo_region_filter,
        alt.value("#f5f5f5"),
        alt.value("#eeeeee")
    ),
    opacity=alt.condition(
        geo_region_filter,
        alt.value(1),
        alt.value(0.25)
    ),
    tooltip=[
        alt.Tooltip("properties.Community Name:N", title="Community Area")
    ]
)

crime_points = alt.Chart(df_map).mark_circle(size=18).encode(
    longitude="Longitude:Q",
    latitude="Latitude:Q",
    color=alt.Color(
        "Primary Type:N",
        title="Crime type",
        scale=alt.Scale(scheme="tableau10")
    ),
    opacity=alt.condition(
        crime_type_select,
        alt.value(0.65),
        alt.value(0.05)
    ),
    tooltip=[
        alt.Tooltip("Date:N", title="Date"),
        alt.Tooltip("Primary Type:N", title="Crime type"),
        alt.Tooltip("Community Name:N", title="Community area")
    ]
).transform_filter(
    region_filter
).transform_filter(
    time_brush
)

crime_map = alt.layer(
    background_map,
    crime_points
).project(
    type="mercator"
).properties(
    width=850,
    height=500,
    title="Where reported crimes occurred in Chicago"
)

monthly_trend = alt.Chart(trend_data).mark_line(point=True).encode(
    x=alt.X(
        "YearMonth:T",
        title="Month",
        axis=alt.Axis(format="%b %Y", labelAngle=-45)
    ),
    y=alt.Y(
        "Reports:Q",
        title="Number of reported crimes"
    ),
    color=alt.Color(
        "Primary Type:N",
        title="Crime type",
        scale=alt.Scale(scheme="tableau10")
    ),
    opacity=alt.condition(
        crime_type_select,
        alt.value(1),
        alt.value(0.12)
    ),
    tooltip=[
        alt.Tooltip("YearMonth:T", title="Month", format="%B %Y"),
        alt.Tooltip("Primary Type:N", title="Crime type"),
        alt.Tooltip("Community Name:N", title="Community area"),
        alt.Tooltip("Reports:Q", title="Reports")
    ]
).add_params(
    time_brush,
    crime_type_select
).transform_filter(
    region_filter
).properties(
    width=850,
    height=300,
    title="Drag across this timeline to focus the map on a specific period"
)

central_interactive_chart = (
    crime_map & monthly_trend
).add_params(
    region_select
).resolve_scale(
    color="shared"
)

central_interactive_chart.save("central_interactive_chicago_crime.html")
central_interactive_chart

TypeError: to_json() got an unexpected keyword argument 'data_dir'

alt.VConcatChart(...)

In [24]:
# =========================
# Contextual visualization 1: top crime types citywide
# GitHub-safe version using aggregated data
# =========================

top_type_data = (
    df_top
    .groupby("Primary Type")
    .size()
    .reset_index(name="Reports")
    .sort_values("Reports", ascending=False)
)

context_top_types = alt.Chart(top_type_data).mark_bar().encode(
    x=alt.X(
        "Reports:Q",
        title="Number of reported crimes"
    ),
    y=alt.Y(
        "Primary Type:N",
        sort="-x",
        title="Crime type"
    ),
    color=alt.Color(
        "Primary Type:N",
        legend=None,
        scale=alt.Scale(scheme="tableau10")
    ),
    tooltip=[
        alt.Tooltip("Primary Type:N", title="Crime type"),
        alt.Tooltip("Reports:Q", title="Reports")
    ]
).properties(
    width=800,
    height=360,
    title="The most common reported crime types in the past year"
)

context_top_types.save("context_top_crime_types.html")
context_top_types

TypeError: to_json() got an unexpected keyword argument 'data_dir'

alt.Chart(...)

In [25]:
# =========================
# Contextual visualization 2: day and hour heatmap
# GitHub-safe version using aggregated data
# =========================

heatmap_data = (
    df_top
    .groupby(["DayOfWeek", "Hour"])
    .size()
    .reset_index(name="Reports")
)

context_heatmap = alt.Chart(heatmap_data).mark_rect().encode(
    x=alt.X(
        "Hour:O",
        title="Hour of day"
    ),
    y=alt.Y(
        "DayOfWeek:N",
        sort=day_order,
        title="Day of week"
    ),
    color=alt.Color(
        "Reports:Q",
        title="Number of reports",
        scale=alt.Scale(scheme="yelloworangered")
    ),
    tooltip=[
        alt.Tooltip("DayOfWeek:N", title="Day of week"),
        alt.Tooltip("Hour:O", title="Hour"),
        alt.Tooltip("Reports:Q", title="Reports")
    ]
).properties(
    width=800,
    height=320,
    title="When reported crimes are concentrated during the week"
)

context_heatmap.save("context_day_hour_heatmap.html")
context_heatmap

TypeError: to_json() got an unexpected keyword argument 'data_dir'

alt.Chart(...)